In [1]:
"""
Plot PSTH (sm fr) for visual presentation data

For tutorial for how to use neuralplot class, see 
/home/lucas/code/neuralplot/neuralplot_nb_LT_260513.ipynb

Steps for analysis here:
1. Run local preprocessing: ./load_and_save_locally_good.sh Pancho 260603 260603
    Make sure you have edited rec_to_beh_Diego.yaml and directory.py to tell it which sessions are the beh ones (see how it's done within)
2. Extract dfallpa (/home/lucas/code/neuralmonkey/neuralmonkey/scripts/analy_dfallpa_extract.sh)
3. Run the below to make all sm fr plot 
    (Note: I previously made rasters using scripts/analy_rasters_script_wrapper_SP.sh, but that's not necessary)
4. For fixation behavior:
    Move to /lemur2/lucas/analyses/recordings/main/MIRROR/Visual/data/Pancho/260603/behavior_fixation/260603_124614_primsmiscfix1_Pancho_1.bhv2
    Then convert to .mat file using /lemur2/lucas/analyses/recordings/main/MIRROR/Visual/convert_bhv2mat.m [run this in MATLAB by hand. it's quick]

"""

"\nPlot PSTH (sm fr) for visual presentation data\n\nFor tutorial for how to use neuralplot class, see \n/home/lucas/code/neuralplot/neuralplot_nb_LT_260513.ipynb\n\nSteps for analysis here:\n1. Run local preprocessing: ./load_and_save_locally_good.sh Pancho 260603 260603\n    Make sure you have edited rec_to_beh_Diego.yaml and directory.py to tell it which sessions are the beh ones (see how it's done within)\n2. Extract dfallpa (/home/lucas/code/neuralmonkey/neuralmonkey/scripts/analy_dfallpa_extract.sh)\n3. Run the below to make all sm fr plot \n    (Note: I previously made rasters using scripts/analy_rasters_script_wrapper_SP.sh, but that's not necessary)\n4. For fixation behavior:\n    Move to /lemur2/lucas/analyses/recordings/main/MIRROR/Visual/data/Pancho/260603/behavior_fixation/260603_124614_primsmiscfix1_Pancho_1.bhv2\n    Then convert to .mat file using /lemur2/lucas/analyses/recordings/main/MIRROR/Visual/convert_bhv2mat.m [run this in MATLAB by hand. it's quick]\n\n"

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from scipy.io import loadmat
import os
import numpy as np
import pandas as pd
from neuralplot import loadNeuralplot, Neuralplot
import tdt
import matplotlib.pyplot as plt

### Load data for given animal/date 
(Diego = Subj. 1, Pacho = Subj. 2)

### Check out data structures
- ```nplot.prettyBeh``` is the data loaded out of monkey logic

- ```nplot.prettyTdt``` is the raw data from our tdt recording system
    - This includes LFP and raw voltage signal but these are not included for sake of file sizes

- ```nplot.Dat``` is the merged dataframe, pulls information from Beh and combines it with timing precision in Tdt

- ```nplot.SpikeTimes``` is dataframe with spike times loaded from kilosort. These have been automatically clustered by ks and manually curated for 'questionable' spikes.

** All timing relating things are handled automatically to align event codes with kilosort spike times. Note that 'photodiode_time' in Tdt/Dat is the most correct timing to align with kilosort. 

** 'sample_index' is also the most accurate indexing variable, with one 'sample_index' per sample that the monkey actually saw (whether they held fixation or not). Sample index DOES NOT include cases where no sample was shown (fixation_success tracks whether he held fixation or not). In the tdt data you will see instances of this where it looks like 'fix_cue' followed immediately by 'sample_off' and 'timeout'. The fact that it says 'sample_off' (even with no 'sample_on') is just an unintended part of the monkeylogic code (i.e. it runs toggleobject(sample,'status','off') whether or not there is a toggleobject(sample,'status,'on)).

In [4]:
map_anidate_to_prims = {
    ("Pancho", 260310): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-2-0", "arcdeep-4-1-0", "arcdeep-4-4-0", "line-8-1-0", "line-8-2-0", "squiggle3-3-2-0"],
    ("Diego", 260304): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-3-0", "arcdeep-4-2-0", "arcdeep-4-3-0", "circle-6-1-0", "line-8-1-0", "line-8-3-0", "squiggle3-3-1-1", "squiggle3-3-2-1"],
    ("Diego", 260306): ["Lcentered-4-1-0", "Lcentered-4-3-0", "V-2-2-0", "V-2-4-0", "arcdeep-4-4-0", "line-8-2-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-0"],
    ("Pancho", 260305): ["Lcentered-4-3-0", "V-2-1-0", "V-2-4-0", "arcdeep-4-2-0", "circle-6-1-0", "line-8-3-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-1"],

    ("Diego", 260529): ["Lcentered-4-1-0", "Lcentered-4-3-0", "V-2-2-0", "V-2-4-0", "arcdeep-4-4-0", "line-8-2-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-0"],
    ("Diego", 260601): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-3-0", "arcdeep-4-2-0", "arcdeep-4-3-0", "circle-6-1-0", "line-8-1-0", "line-8-3-0", "squiggle3-3-1-1", "squiggle3-3-2-1"],

    ("Pancho", 260602): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-2-0", "arcdeep-4-1-0", "arcdeep-4-4-0", "line-8-1-0", "line-8-2-0", "squiggle3-3-2-0"],
    ("Pancho", 260603): ["Lcentered-4-3-0", "V-2-1-0", "V-2-4-0", "arcdeep-4-2-0", "circle-6-1-0", "line-8-3-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-1"],
}


In [5]:
# Just a subset
map_anidate_to_prims = {
    ("Diego", 260529): ["Lcentered-4-1-0", "Lcentered-4-3-0", "V-2-2-0", "V-2-4-0", "arcdeep-4-4-0", "line-8-2-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-0"],
    ("Diego", 260601): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-3-0", "arcdeep-4-2-0", "arcdeep-4-3-0", "circle-6-1-0", "line-8-1-0", "line-8-3-0", "squiggle3-3-1-1", "squiggle3-3-2-1"],

    ("Pancho", 260602): ["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-2-0", "arcdeep-4-1-0", "arcdeep-4-4-0", "line-8-1-0", "line-8-2-0", "squiggle3-3-2-0"],
    ("Pancho", 260603): ["Lcentered-4-3-0", "V-2-1-0", "V-2-4-0", "arcdeep-4-2-0", "circle-6-1-0", "line-8-3-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-1"],
}


In [6]:
# NOTE: confirmed by eye, this is all 17 for Pancho and all 26 for Diego (it includes the ZZ and U)
map_animal_to_shapes_learned = {
    "Diego":["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-3-0", "arcdeep-4-2-0", "arcdeep-4-3-0", "circle-6-1-0", "line-8-1-0", "line-8-3-0", "squiggle3-3-1-1", "squiggle3-3-2-1", "Lcentered-4-1-0", "Lcentered-4-3-0", "V-2-2-0", "V-2-4-0", "arcdeep-4-4-0", "line-8-2-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-0", "usquare-1-2-0", "usquare-1-3-0", "usquare-1-4-0", "zigzagSq-1-1-0", "zigzagSq-1-1-1", "zigzagSq-1-2-0", "zigzagSq-1-2-1"],
    "Pancho":["Lcentered-4-2-0", "Lcentered-4-4-0", "V-2-2-0", "arcdeep-4-1-0", "arcdeep-4-4-0", "line-8-1-0", "line-8-2-0", "squiggle3-3-2-0", "Lcentered-4-3-0", "V-2-1-0", "V-2-4-0", "arcdeep-4-2-0", "circle-6-1-0", "line-8-3-0", "line-8-4-0", "squiggle3-3-1-0", "squiggle3-3-2-1"],
}

In [ ]:
# for animal, dates_list in date_animal_dict.items():
#     for date in dates_list:
#         print(f'###### Doing {animal}, {date} ######')
#         nplot = loadNeuralPlot(animal,date)

# animal = 'Diego'
# date = 260306
# animal = 'Pancho'
# date = 260310


from neuralmonkey.classes.population_mult import load_handsaved_wrapper
from neuralmonkey.classes.population_mult import dfpa_concatbregion_preprocess_wrapper
import seaborn as sns
import os
from neuralplot import extract_neural_data_as_PA
from pythonlib.tools.plottools import savefig


for animal, date in map_anidate_to_prims.keys():
    
    nplot = loadNeuralplot(animal, date)

    ### Preprocessing
    map_stimprefix_to_stimkind = {
        "baseprims_novel_remixes":"drawprims",
        "baseprims_novel_remixes_manual_subset_1":"drawprims",
        "baseprims_only_redlarge":"drawprims", 
        "baseprims_only_bluesmall":"drawprims",
        "omniglot_subset_manual_1":"omniglot",
        "FOB_subsetsmaller_from_Caspar":"FOB",
    }
    def f(x):
        return map_stimprefix_to_stimkind[x]
    nplot.Dat["stimkind_using_stimprefix"] = nplot.Dat["stim_prefix_name"].apply(f)
    nplot.Dat["stimkind_using_stimprefix"].value_counts()

    from pythonlib.tools.pandastools import append_col_with_grp_index
    nplot.Dat = append_col_with_grp_index(nplot.Dat, ["stim_prefix_name", "stim_name"], "stim_prefix_name_combined", False)

    ###
    SAVEDIR = f"/lemur2/lucas/analyses/recordings/main/MIRROR/Visual/smfr/{animal}-{date}"
    os.makedirs(SAVEDIR, exist_ok=True)
    
    ### [LT] Getting monkeylogic files for the shape kind etc
    # from neuralplot import lt_map_stimname_to_actual_shape_params
    # nplot.Dat = lt_map_stimname_to_actual_shape_params(nplot.Dat)
    from neuralplot import lt_map_stimname_to_actual_shape_params_wrapper
    lt_map_stimname_to_actual_shape_params_wrapper(nplot)

    if False: # Dan's code
        #Plot PSTH
        conditions_plot = list(range(1,50))
        params = {
            'condition': conditions_plot, #plot only for these conditions
            'fixation_success_binary': [True] #only plot when fixation is successful
            #You can filter by any column/value pair here, as long as the column is present in 'Dat'
        }
        channel = 41
        fig_dict = nplot.plotPSTH(channel, params, group_by='fixation_success_binary', window=(0.1,0.2))

    ### Plot n trials per unique stim
    # from pythonlib.tools.pandastools import append_col_with_grp_index
    nplot.Dat = append_col_with_grp_index(nplot.Dat, ["stim_prefix_name", "stim_name"], "stim_prefix_name_combined_str")
    df = nplot.Dat.sort_values("stim_prefix_name_combined_str")
    fig = sns.displot(data=df, x="stim_prefix_name_combined_str", aspect=6)
    from pythonlib.tools.snstools import rotateLabel
    rotateLabel(fig)
    savefig(fig, f"{SAVEDIR}/counts_per_unique_stim.pdf")
    plt.close("all")
    
    #### Extract dfallpa, and use that to map the ks units between visual and motor sessions

    # Load a single DFallPA
    version = "trial"
    combine = True
    question = "SP_BASE_trial"
    DFallpa = load_handsaved_wrapper(animal, int(date), version=version, combine_areas=combine, 
                                        question=question)

    dfpa_concatbregion_preprocess_wrapper(DFallpa, animal, date, fr_mean_subtract_method="raw_fr")

    # # Get kilosort metadata
    # DFallpa[:2]
    # PA = DFallpa["pa"].values[0]
    # PA.Xlabels["chans"]
    ### Take Dan visual data, map those units to the unit IDs in PA.
    # First, get dataframe that concats all the brain areas in PA
    dfallpa = DFallpa[DFallpa["event"] == "03_samp"]

    from neuralplot import identify_unit_in_visual_data_using_pa_chans
    identify_unit_in_visual_data_using_pa_chans(nplot, dfallpa)
    
    map_bregion_to_sites = {}
    for bregion in nplot.spikeTimes["bregion"].unique():
        df = nplot.spikeTimes[nplot.spikeTimes["bregion"] == bregion]
        map_bregion_to_sites[bregion] = df["site_final"].unique().tolist()

    ### Get sites
    map_bregioncombined_to_sites = {}
    for bregion_combined in nplot.spikeTimes["bregion_combined"].unique():
        df = nplot.spikeTimes[nplot.spikeTimes["bregion_combined"] == bregion_combined]
        map_bregioncombined_to_sites[bregion_combined] = df["site_final"].unique().tolist()

    shapes_draw = map_anidate_to_prims[(animal, int(date))]
    shapes_learned = map_animal_to_shapes_learned[animal]


    window = (-0.4, 1.0)
    # list_site = [1158, 1161]
    # # list_site = [1158]
    # list_site = [1042, 1044, 1068]


    if False: # Skip this since it is v1
        #### v1 --> Only Visual presentation data. No quantificaiton, just sm fr
        savedir = f"{SAVEDIR}/each_unit"
        os.makedirs(savedir, exist_ok=True)
        for bregion, list_site in map_bregion_to_sites.items():
            print("For this region gettign sitse: ", bregion, list_site)
            PAdan = extract_neural_data_as_PA(nplot, window, list_site, shapes_draw)
            
            ### Plot, split in different ways
            # PAdanprune = PAdan.slice_by_labels_filtdict({"shape":shapes_draw, "shapekind":["baseprims"]})
            
            for chan in PAdan.Chans:
                
                fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "prim", ["shapekind2"], add_x_zero_line=True, size=6, 
                    global_legend=False)
                savefig(fig, f"{savedir}/{chan}-{bregion}-1.pdf")

                fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shapekind2", ["dummy"], add_x_zero_line=True, size=6, 
                    global_legend=False)
                savefig(fig, f"{savedir}/{chan}-{bregion}-2.pdf")

                fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shape", ["shapekind2"], add_x_zero_line=True, size=6, 
                    global_legend=False, add_legend=False)
                savefig(fig, f"{savedir}/{chan}-{bregion}-3.pdf")

                plt.close("all")
    
    #### v2 --> Visual + drawing combined. Also does quantificaiton, including of scalar values. [GOOD]
    from neuralplot import extract_pa_combining_visual_and_draw

    ### Get sites
    twind_scal_early = [0.05, 0.35]
    twind_scal_late = [0.4, 0.75]
    RES = []
    for bregion in map_bregioncombined_to_sites.keys():

        PAall = extract_pa_combining_visual_and_draw(nplot, DFallpa, map_bregioncombined_to_sites, 
            bregion, window, shapes_draw, shapes_learned)

        
        from pythonlib.tools.pandastools import grouping_print_n_samples
        dflab = PAall.Xlabels["trials"]
        grouping_print_n_samples(dflab, ["taskcondition", "shapekind2", "shapekind", "shape"], 
                                 savepath = f"{SAVEDIR}/counts-{bregion}.txt")
        
        ### Get variants
        # Remove draw_stroke, to more directly compare just visual periods
        PAallnostrok = PAall.slice_by_labels_filtdict({"taskcondition":["visual", "draw_plan"]})
        PAvis = PAall.slice_by_labels_filtdict({"taskcondition":["visual"]})
        PAscal_early = PAall.slice_by_dim_values_wrapper("times", twind_scal_early).agg_wrapper("times")
        PAscal_late = PAall.slice_by_dim_values_wrapper("times", twind_scal_late).agg_wrapper("times")
        PAall.Xlabels["trials"]["dummy"] = "dummy"

        for chan in PAall.Chans:

            ##### SMOOTHED FR (combine visual and draw)
            savedir = f"{SAVEDIR}/each_unit_smfr_combine_vis_draw"
            os.makedirs(savedir, exist_ok=True)    

            # Plot all shape
            fig = PAall.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shape", ["tkx_shkind"], add_x_zero_line=True, size=6, 
                    global_legend=False, add_legend=False)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1.pdf")

            # Plot all shape
            fig = PAallnostrok.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shape", ["tkx_shkind"], add_x_zero_line=True, size=6, 
                    global_legend=False, add_legend=False)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1NOSTROK.pdf")

            # Compare conditions in a single plot
            fig = PAall.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "tkx_shkind", ["dummy"], add_x_zero_line=True, size=6, 
                    global_legend=False)
            savefig(fig, f"{savedir}/{chan}-{bregion}-2.pdf")

            # Plot shapes just comparing the base_prims
            pa = PAall.slice_by_labels_filtdict({"shapekind":["baseprims"]})
            fig = pa.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "shape", ["tkx_shkind"], add_x_zero_line=True, size=6, 
                    global_legend=False)
            savefig(fig, f"{savedir}/{chan}-{bregion}-3.pdf")

            plt.close("all")
            
            # Heatmap -- Plot as rows, each stim
            list_twind_scal_eucl = None
            list_mean_zscore_base = [(False, False, False), (False, False, True)]
            pa = PAall.slice_by_dim_values_wrapper("chans", [chan])
            pa.plot_heatmap_state_euclidean_wrapper("shape", "tkx_shkind", savedir, list_twind_scal_eucl, [], [],
                list_mean_zscore_base=list_mean_zscore_base, do_state_space=False, do_euclidean=False, 
                heatmap_plot_fancy = False, heatmap_plot_y_neuron=False, heatmap_plot_nogrouping=False,
                heatmap_subplot_height_by_nrows=True,
                heatmap_save_suffix=f"{chan}-{bregion}")

            pa = PAallnostrok.slice_by_dim_values_wrapper("chans", [chan])
            pa.plot_heatmap_state_euclidean_wrapper("shape", "tkx_shkind", savedir, list_twind_scal_eucl, [], [],
                list_mean_zscore_base=list_mean_zscore_base, do_state_space=False, do_euclidean=False, 
                heatmap_plot_fancy = False, heatmap_plot_y_neuron=False, heatmap_plot_nogrouping=False,
                heatmap_subplot_height_by_nrows=True,
                heatmap_save_suffix=f"{chan}-{bregion}-NOSTROK")

            pa = PAvis.slice_by_dim_values_wrapper("chans", [chan])
            pa.plot_heatmap_state_euclidean_wrapper("shape", "tkx_shkind", savedir, list_twind_scal_eucl, [], [],
                list_mean_zscore_base=list_mean_zscore_base, do_state_space=False, do_euclidean=False, 
                heatmap_plot_fancy = False, heatmap_plot_y_neuron=False, heatmap_plot_nogrouping=False,
                heatmap_subplot_height_by_nrows=True,
                heatmap_save_suffix=f"{chan}-{bregion}-VIS")

            plt.close("all")
            
            ##### SCALAR Quantification
            savedir = f"{SAVEDIR}/each_unit_scalar"
            os.makedirs(savedir, exist_ok=True)

            from pythonlib.tools.pandastools import aggregGeneral
            from pythonlib.tools.snstools import rotateLabel
            from pythonlib.tools.pandastools import plot_45scatter_means_flexible_grouping
            from pythonlib.tools.pandastools import pivot_table
            from pythonlib.tools.statstools import statsmodel_ols
            from pythonlib.tools.regressiontools import fit_and_score_regression_with_categorical_predictor
            from pythonlib.tools.plottools import set_axis_lims_square_bounding_data_45line


            ### Get data for this unit
            # Extract fr data for this unit
            dflab = PAall.Xlabels["trials"].copy()
            frvec_early = PAscal_early.slice_by_dim_values_wrapper("chans", [chan]).X.squeeze()
            frvec_late = PAscal_late.slice_by_dim_values_wrapper("chans", [chan]).X.squeeze()
            dflab["frscal_early"] = frvec_early
            dflab["frscal_late"] = frvec_late
            dflab["frscal"] = (frvec_early + frvec_late)/2
            dflab_agg = aggregGeneral(dflab, ["tkx_shkind", "shape", "taskcondition"], ["frscal_early", "frscal_late", "frscal"])


            ### Q: Differnet firing rates for different kinds of stimuli? E.g., draw vs. novel
            # 1d histogram of firing rates
            order = sorted(dflab["tkx_shkind"].unique())

            fig = sns.catplot(data=dflab_agg, x="tkx_shkind", y="frscal", alpha=0.5, jitter=True, order=order, hue="taskcondition")
            for ax in fig.axes.flatten():
                ax.axhline(0, color="k", alpha=0.5)
            rotateLabel(fig)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_hist_fr-1.pdf")

            fig = sns.catplot(data=dflab_agg, x="tkx_shkind", y="frscal", kind="bar", errorbar="se", order=order, hue="taskcondition")
            rotateLabel(fig)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_hist_fr-2.pdf")

            fig = sns.catplot(data=dflab_agg, x="tkx_shkind", y="frscal", kind="boxen", order=order, hue="taskcondition")
            rotateLabel(fig)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_hist_fr-3.pdf")

            # 2d plots
            fig = sns.relplot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind", alpha=0.5, marker="o")
            for ax in fig.axes.flatten():
                set_axis_lims_square_bounding_data_45line(ax, alpha=0.25)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-1.pdf")

            fig = sns.relplot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind", alpha=0.5, marker="o", col="taskcondition")
            for ax in fig.axes.flatten():
                set_axis_lims_square_bounding_data_45line(ax, alpha=0.25)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-2.pdf")

            if False: # Not interpretable if use hue
                fig = sns.displot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind", col="taskcondition")
                savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-3.pdf")

                fig = sns.displot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind")
                savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-4.pdf")

            fig = sns.displot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind", kind="kde", col="taskcondition")
            for ax in fig.axes.flatten():
                set_axis_lims_square_bounding_data_45line(ax, alpha=0.25)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-5.pdf")

            fig = sns.displot(data=dflab_agg, x="frscal_early", y="frscal_late", hue="tkx_shkind", kind="kde")
            for ax in fig.axes.flatten():
                set_axis_lims_square_bounding_data_45line(ax, alpha=0.25)
            savefig(fig, f"{savedir}/{chan}-{bregion}-1_2dhist_fr-6.pdf")


            ### Q: Correlation to test mirroring between motor and visual
            var_value = "frscal"
            xvar = "draw_plan|baseprims|1|1"
            yvar = "visual|baseprims|1|1"
            xvar_fr = f"frscal-{xvar}"
            yvar_fr = f"frscal-{yvar}"

            # Scatter
            dfres, fig = plot_45scatter_means_flexible_grouping(dflab, "tkx_shkind", 
                xvar, yvar,  None, var_value, "shape", True, SIZE=5)
            assert dfres is not None
            dfres_wide = pivot_table(dfres, ["shape"], ["tkx_shkind"], ["frscal"], flatten_col_names=True)

            # Regress
            ax = fig.axes[0]
            sns.regplot(data=dfres_wide, x=xvar_fr, y=yvar_fr, ax=ax, marker="none", color="r")

            # Correlation
            # results, intercept, slope, r2, pvalue = statsmodel_ols(dfres_wide[xvar_fr], dfres_wide[yvar_fr], True, ax, overlay_color="r", overlay_font_size=8)
            results, intercept, slope, r2, pvalue = statsmodel_ols(dfres_wide[xvar_fr], dfres_wide[yvar_fr], 
                True, ax, overlay_color="r", overlay_font_size=8, return_stats=True)

            # Save
            savefig(fig, f"{savedir}/{chan}-{bregion}-2_scatter.pdf")

            # Also simply quantify how activity cares about shape within each taskcondition (visual vs draw)
            r2_vals = [r2]
            for tk in ["visual", "draw_plan"]:
                df = dflab[(dflab["tkx_shkind"] == f"{tk}|baseprims|1|1")].reset_index(drop=True)
                dict_coeff, model, original_feature_mapping, results = fit_and_score_regression_with_categorical_predictor(
                    df, "frscal", ["shape"], [True], None, PRINT=False)
                # print(tk, results['r2_train'])
                r2_vals.append(results['r2_train'])

            fig, ax = plt.subplots()
            ax.bar(["visual-vs-draw_plan", "visual", "draw_plan"], r2_vals)
            ax.set_xlabel(["visual-vs-draw_plan", "visual", "draw_plan"])
            ax.set_ylabel("r2")
            ax.set_ylim([0, 1])
            savefig(fig, f"{savedir}/{chan}-{bregion}-2_r2_barplot.pdf")

            ### Store things for this neuron
            RES.append({
                "chan":chan,
                "bregion":bregion,
                "r2_visual_vs_draw":r2_vals[0],
                "intercept":intercept,
                "slope":slope, 
                "pvalue":pvalue,
                "r2_visual":r2_vals[1],
                "r2_draw":r2_vals[2],
            })

            # Save the data for this (neuron)
            dflab.to_pickle(f"{SAVEDIR}/dflab-{chan}.pkl")
            dflab_agg.to_pickle(f"{SAVEDIR}/dflab_agg-{chan}.pkl")

            plt.close("all")    

    # Save RES
    DFRES = pd.DataFrame(RES)
    DFRES.to_pickle(f"{SAVEDIR}/DFRES.pkl")        

Searching using this string:
/home/lucas/mnt/Freiwald/ltian/recordings/*Diego*/*260529*/**
Found this many paths:
6
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-125254
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-133004
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-141111
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-144722
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-151309
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-155249
Searching using this string:
/home/lucas/mnt/Freiwald/ltian/recordings/*Diego*/*260529*/**
Found this many paths:
6
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-125254
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-133004
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/260529/Diego-260529-141111
---
/home/lucas/mnt/Freiwald/ltian/recordings/Diego/26052

/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)
/home/lucas/code/pythonlib/pythonlib/tools/snstools.py:25: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels(list_text,rotation=rotation, horizontalalignment="right")
/home/lucas/miniconda3/envs/drag2_matlab/lib/python3.8/site-packages/seaborn/axisgrid.py:118: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)


Loading DFallpa from:  /lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/Xuan/DFallpa-Diego-260529-trial-kilosort_if_exists-norm=None-combine=True-t1=-1.0-t2=1.8-quest=SP_BASE_trial.pkl
 == (1) Matching chans across events
M1  ...  11
M1  ...  11
M1  ...  11
M1  -- n chans final:  11
PMv  ...  37
PMv  ...  37
PMv  ...  37
PMv  -- n chans final:  37
PMd  ...  27
PMd  ...  27
PMd  ...  27
PMd  -- n chans final:  27
dlPFC  ...  12
dlPFC  ...  12
dlPFC  ...  12
dlPFC  -- n chans final:  12
vlPFC  ...  17
vlPFC  ...  17
vlPFC  ...  17
vlPFC  -- n chans final:  17
FP  ...  26
FP  ...  26
FP  ...  26
FP  -- n chans final:  26
SMA  ...  23
SMA  ...  23
SMA  ...  23
SMA  -- n chans final:  23
preSMA  ...  26
preSMA  ...  26
preSMA  ...  26
preSMA  -- n chans final:  26
 == (2) Remove bad chans based on drift
GOOD!! -- passed all tests, channels match (sitesdirty)
============== REMOVING DIRTY SITES:
... bregion  M1 ... event  03_samp
Removing these bad chans: [1002, 1006]
Chans exist in PA: [1000

In [ ]:
### Debugging.

In [ ]:
from neuralplot import extract_neural_data_as_PA

list_site = [1168]
PAvis = extract_neural_data_as_PA(nplot, window, list_site, shapes_draw, shapes_learned)


In [ ]:
dflab = PAvis.Xlabels["trials"]
dflab["shapekind2"].value_counts()

In [ ]:
# Making nice heatmaps of all stimuli

In [ ]:
list_site = map_bregion_to_sites["PMv_m"]
list_site = list_site[:3]
# list_site = [1044]
list_site = [1152]


In [ ]:
list_site
print("For this region gettign sitse: ", bregion, list_site)
PAdan = extract_neural_data_as_PA(nplot, window, list_site, shapes_draw)


In [ ]:
dflab = PAdan.Xlabels["trials"]
dflab[:2]

In [ ]:
# First, order stim how you want them to be plot on the heatmap
from pythonlib.tools.pandastools import append_col_with_grp_index
dflab = PAdan.Xlabels["trials"]
dflab = append_col_with_grp_index(dflab, ["shapekind2", "shape"], "shkind_sh")
PAdan.Xlabels["trials"] = dflab

### Load saved DFRES and plot summary of scalar scores across bregions

In [ ]:

animal = "Pancho"
date = 260305
SAVEDIR = f"/lemur2/lucas/analyses/recordings/main/MIRROR/Visual/smfr/{animal}-{date}"
DFRES = pd.read_pickle(f"{SAVEDIR}/DFRES.pkl")        

In [ ]:
import seaborn as sns

In [ ]:
DFRES[:2]

In [ ]:
from pythonlib.tools.pandastools import convert_wide_to_long
# DFRES_LONG = convert_wide_to_long(DFRES, ["r2_visual_vs_draw", "slope", "r2_visual", "r2_draw"], ["chan", "bregion"], new_col_name_level="variable", new_col_name_value="value")
DFRES_LONG = convert_wide_to_long(DFRES, ["r2_visual_vs_draw", "r2_visual", "r2_draw"], ["chan", "bregion"], new_col_name_level="variable", new_col_name_value="value")
DFRES_LONG[:2]

In [ ]:
fig = sns.catplot(data=DFRES_LONG, col="bregion", y="value", x="variable", alpha=0.5, jitter=True)
for ax in fig.axes.flatten():
    ax.axhline(0, color="k", alpha=0.5)

In [ ]:
fig = sns.catplot(data=DFRES_LONG, col="variable", y="value", x="bregion", alpha=0.5, jitter=True)
for ax in fig.axes.flatten():
    ax.axhline(0, color="k", alpha=0.5)

In [ ]:
fig = sns.catplot(data=DFRES_LONG, x="bregion", y="value", hue="variable", kind="bar")
for ax in fig.axes.flatten():
    ax.axhline(0, color="k", alpha=0.5)

In [ ]:
DFRES[:2]

In [ ]:
sns.relplot(data=DFRES, x="r2_draw", y = "r2_visual", hue="r2_visual_vs_draw",  col="bregion")

In [ ]:
DFRES.loc[:, ["chan", ]]

In [ ]:
sns.heatmap(data=DFRES)

In [ ]:
sns.catplot(data = DFRES, x=)

### In progress

In [ ]:
fig = PAdan.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "stim_name_index", ["shapekind2"], add_x_zero_line=True, size=6, 
    global_legend=False)


for chan in PAdan.Chans:
    fig = PAdanprune.plotwrapper_smoothed_fr_split_by_label_and_subplots(chan, "prim", ["shapekind"])
    from pythonlib.tools.plottools import savefig 
    savefig(fig, f"/tmp/{chan}.png")
    plt.close("all")

PAdan.plotwrappergrid_smoothed_fr_splot_neuron("prim", ["shapekind"], [1001, 1002])


In [ ]:

### Plot the same thing, using PA draw
PA = DFallpa["event"] == ""